# 🧭 Pipeline opérationnel : QRU → diagnostic quaternionique → QAE-compatible registerization → QAOA

Ce notebook montre uniquement les objets nécessaires à l'article.

Objectif : comparer un readout standard $\langle Zangle$ avec un readout $v_{quat}$ choisi à partir du mouvement quaternionique du QRU.

Chaîne étudiée :

$$
U_	heta(x) ightarrow q_	heta(x) ightarrow \delta_q ightarrow v_{quat} ightarrow s_{v_{quat}}(x) ightarrow p_v(x) ightarrow \widehat{s}_v(x) ightarrow H_{QAOA}(h).
$$


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
else:
    sys.path.insert(0, str(ROOT.parent / 'src'))

from qru_registerization.gates import random_qru_params
from qru_registerization.pipeline import compute_paths
from qru_registerization.quaternion_diagnostics import (
    quaternion_guided_axis,
    quaternion_motion,
    readout_values,
    readout_variations,
    local_hidden_motion_ratio,
    hidden_motion_ratio,
    quaternion_weighted_energy,
)
from qru_registerization.readout_estimators import shot_estimate_signed_coordinate, ideal_qae_register_estimate
from qru_registerization.qaoa_toy import evaluate_qaoa_with_estimated_field


## 1. Construire une trajectoire QRU

Un QRU single-qubit prépare :

$$
|\psi_	heta(x)angle = U_	heta(x)|0angle, \qquad U_	heta(x)\in SU(2).
$$

On calcule deux trajectoires :

$$
q_	heta(x)\in S^3
$$

pour le mouvement quaternionique de l'unitaire, et

$$
r_	heta(x)=(\langle Xangle,\langle Yangle,\langle Zangle)^T
$$

pour le mouvement observable sur la sphère de Bloch.


In [ ]:
xs = np.linspace(-np.pi, np.pi, 161)
params = random_qru_params(depth=5, seed=13, scale=0.9)
paths = compute_paths(params, xs)
Q = paths['quaternions']
R = paths['bloch_direct']
states = paths['states']

print('quaternions:', Q.shape)
print('Bloch path:', R.shape)
print('max ||r_direct-r_quat||:', np.max(np.linalg.norm(paths['bloch_direct'] - paths['bloch_quaternion'], axis=1)))


## 2. Diagnostiquer le mouvement caché

Le mouvement interne du QRU est mesuré par :

$$
\delta_q(i)=d_{S^3}(q_i,q_{i+1})=2rccos(|q_i\cdot q_{i+1}|).
$$

Un readout scalaire $s_v(x)=v\cdot r_	heta(x)$ ne voit que :

$$
\delta_v(i)=|s_v(x_{i+1})-s_v(x_i)|.
$$

Si $\delta_q(i)$ est grand mais $\delta_v(i)$ petit, alors le readout est localement aveugle.


In [ ]:
ez = np.array([0.0, 0.0, 1.0])
v_quat = quaternion_guided_axis(R, Q, xs)
print('v_quat =', v_quat)

dq = quaternion_motion(Q)
dz = readout_variations(R, ez)
dvq = readout_variations(R, v_quat)
x_mid = 0.5*(xs[:-1]+xs[1:])

fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axs[0].plot(x_mid, dq, color='black', label=r'$\delta_q$')
axs[0].set_ylabel('mouvement dans $S^3$')
axs[0].legend()
axs[1].plot(x_mid, dz, label=r'$\delta_{e_z}$')
axs[1].plot(x_mid, dvq, label=r'$\delta_{v_{quat}}$')
axs[1].set_xlabel('$x$')
axs[1].set_ylabel('variation visible')
axs[1].legend()
plt.tight_layout()
plt.show()


## 3. Comparer les axes fixes et $v_{quat}$

L'axe $v_{quat}$ maximise l'énergie :

$$
E_{quat}(v)=\sum_i \delta_q(i)^2 (v\cdot \Delta r_i)^2.
$$

Il est donc optimal pour l'objectif précis : capturer les variations de Bloch situées dans les régions où l'unitaire bouge fortement.


In [ ]:
axes = {
    'e_x': np.array([1.0,0.0,0.0]),
    'e_y': np.array([0.0,1.0,0.0]),
    'e_z': np.array([0.0,0.0,1.0]),
    'v_quat': v_quat,
}
energies = {name: quaternion_weighted_energy(R, Q, v, xs) for name, v in axes.items()}
hidden = {name: hidden_motion_ratio(R, Q, v) for name, v in axes.items()}
print('Quaternion-weighted energies:', energies)
print('Hidden-motion ratios:', hidden)

fig, axs = plt.subplots(1, 2, figsize=(11,4))
axs[0].bar(energies.keys(), energies.values())
axs[0].set_title('$E_{quat}(v)$')
axs[0].set_ylabel('higher is better')
axs[1].bar(hidden.keys(), hidden.values())
axs[1].set_title('global hidden-motion ratio')
axs[1].set_ylabel('lower is better')
plt.tight_layout(); plt.show()


## 4. Interface QAE-compatible

Après sélection de $v_{quat}$, on lit :

$$
s_{v_{quat}}(x)=v_{quat}\cdot r_	heta(x).
$$

On convertit en probabilité :

$$
p_v(x)=rac{1+s_v(x)}{2}.
$$

Un modèle QAE idéal donne $\widetilde p_v$ avec $|\widetilde p_v-p_v|\leq \epsilon_p$, puis :

$$
\widetilde s_v = 2\widetilde p_v-1.
$$

Enfin on quantifie en registre signed-magnitude $m$ bits.


In [ ]:
s_quat = readout_values(R, v_quat)
for m in [4, 6, 8, 10]:
    estimates = [ideal_qae_register_estimate(s, epsilon_p=1/256, m_bits=m, seed=i)['s_quant'] for i, s in enumerate(s_quat)]
    err = np.abs(np.array(estimates)-s_quat)
    print(f'm={m}: mean error={err.mean():.4e}, max error={err.max():.4e}')

plt.figure(figsize=(8,4))
plt.plot(xs, s_quat, label=r'true $s_{v_{quat}}$')
estimates = np.array([ideal_qae_register_estimate(s, epsilon_p=1/256, m_bits=8, seed=i)['s_quant'] for i, s in enumerate(s_quat)])
plt.plot(xs, estimates, '--', label='QAE-compatible register model, m=8')
plt.xlabel('$x$'); plt.ylabel('signed readout')
plt.legend(); plt.tight_layout(); plt.show()


## 5. Cas aval : QRU $ightarrow$ QAOA toy layer

On utilise le readout QRU comme champ local :

$$
H(h)=-Z_0Z_1-hZ_0-0.35Z_1.
$$

Baseline : mesurer $\langle Zangle$ avec des shots, puis réinjecter $h$ classiquement.

Méthode : utiliser $v_{quat}$, convertir via $p_v=(1+s_v)/2$, puis utiliser le modèle QAE-compatible registerized.


In [ ]:
rng = np.random.default_rng(2026)
idxs = np.linspace(0, len(xs)-1, 15, dtype=int)
z_readout = readout_values(R, ez)
rows = []
for i in idxs:
    h_true = float(s_quat[i])
    h_z = shot_estimate_signed_coordinate(float(z_readout[i]), shots=64, seed=int(rng.integers(1_000_000)))
    h_qae = ideal_qae_register_estimate(h_true, epsilon_p=1/128, m_bits=9, seed=int(rng.integers(1_000_000)))['s_quant']
    out_z = evaluate_qaoa_with_estimated_field(h_true, h_z, grid_size=21)
    out_q = evaluate_qaoa_with_estimated_field(h_true, h_qae, grid_size=21)
    rows.append(('measure_Z_reinject', h_true, h_z, out_z['energy_gap_to_ground'], out_z['ground_state_probability']))
    rows.append(('vquat_QAE_register', h_true, h_qae, out_q['energy_gap_to_ground'], out_q['ground_state_probability']))

for strategy in ['measure_Z_reinject','vquat_QAE_register']:
    rr=[r for r in rows if r[0]==strategy]
    print(strategy)
    print('  mean |h_est-h_true| =', np.mean([abs(r[2]-r[1]) for r in rr]))
    print('  mean energy gap     =', np.mean([r[3] for r in rr]))
    print('  mean ground prob.   =', np.mean([r[4] for r in rr]))
